In [ ]:
MODEL_NAME  = 'vinai/phobert-base'
NER_MODEL   = 'NlpHUST/ner-vietnamese-electra-base'
MAX_LEN     = 256
HIDDEN_DIM  = 768
HEAD_DIM    = HIDDEN_DIM

BATCH_SIZE      = 16
NUM_EPOCHS      = 8
LR              = 2e-5
HEAD_LR         = 5e-5
WARMUP_RATIO    = 0.06
DROPOUT         = 0.1
PATIENCE        = 2
WEIGHT_DECAY    = 0.01
LABEL_SMOOTHING = 0.05
GRAD_ACCUM      = 2       # effective batch = 32
ALPHA_CL        = 0.1    # contrastive loss weight

NUM_CLASSES = 2
LABEL_NAMES = ['NON-HATE', 'HATE']
CKPT_NAME   = 'best_viamplehate_phobert_vihsd.pt'
PLOT_TITLE  = 'AmpleHate-Vi++ (PhoBERT) — ViHSD Proposed'

In [ ]:
import os, re, time, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, classification_report, confusion_matrix
)
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup, pipeline

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PIN_MEMORY = DEVICE.type == 'cuda'
NUM_WORKERS = 2 if os.cpu_count() and os.cpu_count() > 2 else 0

print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Workers: {NUM_WORKERS} | Pin memory: {PIN_MEMORY}')

In [ ]:
!pip install transformers datasets sentencepiece huggingface_hub underthesea easydict -q

# AmpleHate-Vi++ on ViHSD: PhoBERT Proposed

Implements **AmpleHate-Vi++** — Vietnamese-adapted AmpleHate with:
1. Vietnamese NER (`NlpHUST/ner-vietnamese-electra-base`) + target cue lexicon → coverage 0.09% → ~45%
2. Separate attack cue bank (offensive predicates)
3. Relation Bank: 3 HeadAttention modules (r_exp, r_imp, r_atk) fused via Linear
4. Instance-adaptive gate `g = σ(W·[h_CLS; r])` replacing fixed scalar `e`
5. CrossEntropy (weighted) + ContrastiveLoss (α=0.1)
6. max_length=256, gradient accumulation=2, 8 epochs max

**Baseline reference:** `notebooks/models/baselines/ViHSD - Baseline AmpleHate_PhoBERT/`
**Spec:** `docs/superpowers/specs/2026-05-21-viamplehate-proposed-design.md`